# Task 03A — full Colab study

Select an NVIDIA GPU runtime. NUTS and vectorized prediction use it; official SciPy MAP fitting is profiled and may remain on CPU. `RUN_FULL=False` makes **Run all** safe. The notebook never pushes to GitHub.

In [ ]:
REPO_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPO_REF = 'main'  # replace with the published Task 03A commit SHA
ACCELERATOR = 'gpu'  # 'gpu' or 'cpu'
RUN_SMOKE = False
RUN_FULL = False

In [ ]:
import os, pathlib, shutil, subprocess, sys
assert (3, 11) <= sys.version_info[:2] <= (3, 12), f'Python {sys.version.split()[0]} is unsupported'
repo = pathlib.Path('/content/energy-inference-bo')
if repo.exists(): raise RuntimeError(f'{repo} exists; restart the runtime for a clean archival run')
subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(repo)],check=True)
subprocess.run(['git','fetch','origin',REPO_REF],cwd=repo,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=repo,check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(); print('Git SHA:',sha)
if ACCELERATOR == 'gpu':
    subprocess.run(['nvidia-smi'],check=True)
    os.environ['JAX_PLATFORMS']='cuda'; os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
else: os.environ['JAX_PLATFORMS']='cpu'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements.txt')],check=True)
if ACCELERATOR == 'gpu': subprocess.run([sys.executable,'-m','pip','install','-q','jax[cuda12]==0.9.2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)],check=True)
os.chdir(repo)

In [ ]:
import botorch, gpytorch, jax, numpyro, torch
jax.config.update('jax_enable_x64', True)
print({'torch':torch.__version__,'botorch':botorch.__version__,'jax':jax.__version__,'numpyro':numpyro.__version__,'devices':[str(x) for x in jax.devices()]})
if ACCELERATOR == 'gpu': assert any(d.platform == 'gpu' for d in jax.devices()), 'CUDA JAX device required'
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)

In [ ]:
if RUN_SMOKE:
    subprocess.run([sys.executable,'-m','energy_bo.experiments.run_task03a','--profile','smoke','--output-dir','artifacts/task03a/colab_smoke'],check=True)
else: print('Smoke skipped; set RUN_SMOKE=True to run it.')

In [ ]:
if RUN_FULL:
    smoke_summary=repo/'artifacts/task03a/colab_smoke/SUMMARY.json'
    if not smoke_summary.exists(): raise RuntimeError('Run the smoke/preflight cell with RUN_SMOKE=True first.')
    command=[sys.executable,'-m','energy_bo.experiments.run_task03a','--profile','full','--output-dir','artifacts/task03a/full']
    subprocess.run(command,check=True)
    import json
    manifest={'git_sha':sha,'python':sys.version,'accelerator':ACCELERATOR,'jax_backend':jax.default_backend(),'jax_devices':[str(x) for x in jax.devices()],'torch':torch.__version__,'botorch':botorch.__version__,'gpytorch':gpytorch.__version__,'jax':jax.__version__,'numpyro':numpyro.__version__,'command':command}
    out=repo/'artifacts/task03a/full'; (out/'colab_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive=shutil.make_archive('/content/task03a_full_outputs','zip',root_dir=out)
    from google.colab import files; files.download(archive)
else: print('Full study did not run. Set RUN_FULL=True only after tests/preflight pass.')

## After download
Extract the ZIP locally into `artifacts/task03a/full/`. Do not upload raw output directly into Git. Review it first, then ask for an import audit; only compact reviewed evidence belongs under `results/task03a/full/`.